# Reunión 16 — Selección de variables y comparación de clasificadores

Para cada uno de **4 datasets** de entrada:
1. Construir las variables de entrada (4 datasets).
2. Sobre el **conjunto de train**, aplicar **3 métodos de selección de variables** y quedarnos con
   las que aparezcan en **≥ 2 de los 3** (reducimos redundancia → mejores scores).
   - Método 1: **LASSO stability selection**
   - Método 2: **Importancia con Random Forest**
   - Método 3: **RFECV** (eliminación recursiva con validación cruzada)
3. Con el conjunto final de variables, comparar **6 clasificadores** (SVM, RF, LogReg, Árbol, KNN,
   XGBoost) con `class_weight='balanced'`, optimizados con GridSearch, y evaluar en **test**.

> **Decisiones tomadas** (justificadas):
> - **Outcome** = `riesgo_cv_compuesto` ≥ 1 alteración cardíaca **sin strain** (el strain tiene 90% de
>   missings). Con ≥ 2 los positivos serían demasiado escasos para entrenar.
> - **Variables compuestas** (`ant_obstetrico`, `ant_medico`, `trastorno_hipertensivo`, `comp_*`): se
>   reutilizan las **definiciones del equipo en `reunion10`**.
> - **Analítica (Dataset 3)**: se incluyen solo las analíticas con missings asumibles (< ~35%); se
>   descartan troponina T, NT-proBNP (52%) y albúmina/creatinina (39%).
> - **KNN** no admite `class_weight`; se usa `weights='distance'` como equivalente más cercano.
> - Todo (imputación, escalado, **selección**) se ajusta **solo con train**; el test solo se toca al final.


In [1]:
import os, warnings
# Silenciar warnings de forma robusta (incl. los procesos paralelos de joblib/loky):
os.environ['PYTHONWARNINGS'] = 'ignore'   # lo heredan los workers de GridSearch/RFECV (n_jobs=-1)
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
np.seterr(all='ignore')                   # overflow/divide-by-zero de numpy (datos casi separables)

RANDOM_STATE = 42

df  = pd.read_csv('datos.csv');               df['id']  = df['id'].astype(float)
df2 = pd.read_csv('datos_preprocesados.csv'); df2['id'] = df2['id'].astype(float)
print('datos.csv:', df.shape, '| datos_preprocesados.csv:', df2.shape)

datos.csv: (459, 344) | datos_preprocesados.csv: (608, 433)


## 1. Derivación de variables

### 1.1 Variables compuestas (definiciones de `reunion10`)

In [2]:
# Definiciones EXACTAS de reunion10 (no se inventan)
cols_ant_obstetrico = ['parto_previo_menor37_pre','aborto_menor20','ant_cir','ant_peg','ant_obito','ant_pe','ant_hellp']
cols_ant_medico     = ['ant_cesarea','ant_diabetes_pregest','hta_pregest','sindr_antifosfolipido','enf_autoinm','fuma','alcohol','drogas']
cols_hipertension   = ['pe','sd_hellp','hipertension_gest']
cols_placentaria    = ['cir','peg','desprendimiento_placenta']
cols_materna_grave  = ['uci_materna_ucoi','hemocerebral_ictus','trombosis_venosa_prof']
cols_metabolica     = ['diabetes_gest','colestasis_intrahepatica']

def _num(cols):
    return df[cols].apply(pd.to_numeric, errors='coerce')

df['ant_obstetrico']         = (_num(cols_ant_obstetrico).sum(axis=1) >= 1).astype(int)
df['ant_medico']             = (_num(cols_ant_medico).sum(axis=1) >= 1).astype(int)
df['trastorno_hipertensivo'] = _num(cols_hipertension).max(axis=1)
df['comp_placentaria']       = _num(cols_placentaria).max(axis=1)
df['comp_maternaGrave']      = _num(cols_materna_grave).max(axis=1)
df['comp_metabolica']        = _num(cols_metabolica).max(axis=1)

print(df[['ant_obstetrico','ant_medico','trastorno_hipertensivo',
          'comp_placentaria','comp_maternaGrave','comp_metabolica']].sum())

ant_obstetrico            123.0
ant_medico                103.0
trastorno_hipertensivo    132.0
comp_placentaria           69.0
comp_maternaGrave          41.0
comp_metabolica            48.0
dtype: float64


### 1.2 Variables MoM (fórmula de `reunion14`)

Se traen las analíticas angiogénicas crudas de `datos_preprocesados` y se normalizan por edad
gestacional (Múltiplos de la Mediana).

In [3]:
mom_src = ['valor_sflt1','valor_plgf','ratio_sflt1_plgf','eg_deter_sflt1_plgf','creatinina']
df = df.merge(df2[['id'] + mom_src].drop_duplicates('id'), on='id', how='left')

def calcular_mom(d):
    d = d.copy()
    eg = d['eg_deter_sflt1_plgf']
    cond = [(eg>=10)&(eg<15),(eg>=15)&(eg<20),(eg>=20)&(eg<24),
            (eg>=24)&(eg<29),(eg>=29)&(eg<34),(eg>=34)&(eg<37),(eg>=37)]
    div_ratio = [24.8,10.5,4.92,3.06,3.75,9.03,19.6]
    div_sflt1 = [1328,1355,1299,1355,1742,2552,3485]
    div_plgf  = [52.6,135,264,465,471,284,191]
    d['ratio_MoM'] = d['ratio_sflt1_plgf'] / np.select(cond, div_ratio, default=np.nan)
    d['sflt1_MoM'] = d['valor_sflt1']      / np.select(cond, div_sflt1, default=np.nan)
    d['plgf_MoM']  = d['valor_plgf']       / np.select(cond, div_plgf,  default=np.nan)
    return d

df = calcular_mom(df)
print('MoM calculadas. Missings: ratio_MoM %.0f%%, sflt1_MoM %.0f%%, plgf_MoM %.0f%%' % (
    df['ratio_MoM'].isna().mean()*100, df['sflt1_MoM'].isna().mean()*100, df['plgf_MoM'].isna().mean()*100))

MoM calculadas. Missings: ratio_MoM 27%, sflt1_MoM 27%, plgf_MoM 27%


### 1.3 Codificaciones (etnia, concepción, tipo de parto, tomo, hábitos)

- `etnia` → **caucásica (1) vs otra (0)**  ·  `concepcion` → **espontánea (1) vs otra (0)**
- `tipo_parto` → 3 clases (nominal, se hará one-hot)  ·  `tomo` → **tratamiento sí (1) vs no (0)**
- `tabaco` = `fuma_post` (nominal)  ·  `dieta` = `score_dieta`  ·  `ejercicio` = `act_fisica_total`  ·  `estres` = `score_estres`
- `ant_medico_actual` = **propuesta** (≥1 condición crónica actual) — *revisar con clínica*.

In [4]:
df['etnia_caucasica']       = (df['etnia'] == 'Blanca').astype(int)
df['concepcion_espontanea'] = (df['concepcion'] == 'Espontanea').astype(int)
# tipo_parto se deja tal cual (nominal: Cesarea / Eutocico / Instrumentado)

# tomo: 0 = no tomó; 1/2/3 = tomó tratamiento; texto ('Not asked'...) -> NaN
tomo_num = pd.to_numeric(df['tomo_durante_gest'], errors='coerce')
df['tomo'] = np.where(tomo_num.notna(), (tomo_num >= 1).astype(float), np.nan)

df['tabaco']    = df['fuma_post']        # nominal (No / Si / Ex_Fumadora)
df['dieta']     = df['score_dieta']
df['ejercicio'] = df['act_fisica_total']
df['estres']    = df['score_estres']

cols_ant_med_act = ['hta_cronica','dislipemia','diabetes_mellitus_1_2','enf_renal_cronica','enf_autoinm_post','ant_infarto_ictus']
cols_ant_med_act = [c for c in cols_ant_med_act if c in df.columns]
df['ant_medico_actual'] = (df[cols_ant_med_act].apply(pd.to_numeric, errors='coerce').fillna(0).sum(axis=1) >= 1).astype(int)
print('Codificaciones creadas. ant_medico_actual positivos:', int(df['ant_medico_actual'].sum()))

Codificaciones creadas. ant_medico_actual positivos: 87


## 2. Outcome (etiqueta)

`riesgo_cv_compuesto = 1` si ≥ 1 de: TAS≥140 o TAD≥90 · masa VI indexada >95 · FEVI<50 ·
cardiopatía estructural. Se etiqueta sobre las pacientes con esas 5 variables presentes.

In [5]:
outcome_vars = ['ta_sistolica','ta_diastolica','masa_vi_tdiast_indexada','fevi_simpson_4c','cardiopatia_estructural']
coh = df.dropna(subset=outcome_vars).copy()
coh['riesgo_cv_compuesto'] = (
    (coh['ta_sistolica'] >= 140) | (coh['ta_diastolica'] >= 90) |
    (coh['masa_vi_tdiast_indexada'] > 95) | (coh['fevi_simpson_4c'] < 50) |
    (coh['cardiopatia_estructural'] == 1)
).astype(int)
print(f'Cohorte etiquetada: N={len(coh)}, positivos={int(coh.riesgo_cv_compuesto.sum())} '
      f'({coh.riesgo_cv_compuesto.mean()*100:.1f}%)')

Cohorte etiquetada: N=369, positivos=57 (15.4%)


## 3. Definición de los 4 datasets

In [6]:
ds1_embarazo = ['peso_ini_gest','peso_fin_gest','aumento_peso_gest','talla','imc_ini_gest','peso_rn',
    'apgar_1min','apgar_5min','apgar_10min','eg_parto','sflt1_MoM','tas_1tri','tad_1tri','edad_materna_gest',
    'plgf_MoM','ratio_MoM','etnia_caucasica','concepcion_espontanea','tipo_parto','tomo','valor_plgf',
    'ratio_sflt1_plgf','ant_obstetrico','ant_medico','trastorno_hipertensivo','comp_placentaria',
    'comp_maternaGrave','comp_metabolica']

ds2_clasicas = ['edad_actual','peso_actual','imc_actual','ant_medico_actual','tabaco','dieta','ejercicio','estres']

analitica = ['grasa_visceral','ratio_cintura_cadera','hemoglobina_glicada','ldl','hdl','trigliceridos',
             'creatinina','urato_acidourico']   # factibles por % missings (<~35%)
ds3_clasicas_analitica = ds2_clasicas + analitica
ds4_completo = ds1_embarazo + ds3_clasicas_analitica

DATASETS = {'1_embarazo': ds1_embarazo, '2_clasicas': ds2_clasicas,
            '3_clasicas_analitica': ds3_clasicas_analitica, '4_completo': ds4_completo}
NOMINALES = ['tipo_parto', 'tabaco']   # el resto se tratan como numéricas/binarias

for n, cols in DATASETS.items():
    falt = [c for c in cols if c not in coh.columns]
    print(f'Dataset {n}: {len(cols)} variables' + (f'  | NO encontradas: {falt}' if falt else '  | OK'))

Dataset 1_embarazo: 28 variables  | OK
Dataset 2_clasicas: 8 variables  | OK
Dataset 3_clasicas_analitica: 16 variables  | OK
Dataset 4_completo: 44 variables  | OK


## 4. Funciones del pipeline (todo se ajusta solo con train)

In [7]:
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer, SimpleImputer
from sklearn.ensemble import ExtraTreesRegressor, RandomForestClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import RFECV
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score, confusion_matrix
from collections import Counter

def preparar(coh, feature_cols, nominales):
    """
    Réplica de la función 'imputacion_pipeline' de missForest_imp_tfm.ipynb
    con división train/test, imputación en dos pasos, clipping y redondeo.
    """
    cols = [c for c in feature_cols if c in coh.columns]
    nom  = [c for c in nominales if c in cols]
    
    # Identificar numéricas y ordinales (en reunion16 actúan como numéricas)
    cat_cols_ord = [] 
    num = [c for c in cols if c not in nom and c not in cat_cols_ord]
    
    X = coh[cols].copy()
    X[num] = X[num].apply(pd.to_numeric, errors='coerce').astype(float)
    if cat_cols_ord:
        X[cat_cols_ord] = X[cat_cols_ord].apply(pd.to_numeric, errors='coerce').astype(float)
        
    y = coh['riesgo_cv_compuesto'].values
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)

    # Listas de entrenamiento separadas (Equivalentes a train_num, test_num, train_cat, test_cat)
    train_num = Xtr[num]
    test_num  = Xte[num]
    train_cat = Xtr[cat_cols_ord]
    test_cat  = Xte[cat_cols_ord]

    # Diccionarios de mínimos y máximos calculados estrictamente en TRAIN (Evita Data Leakage)
    min_values_dict = Xtr.min().to_dict()
    max_values_dict = Xtr.max().to_dict()

    # IMPUTACIÓN DE NUMÉRICAS
    min_values_num = [min_values_dict[col] for col in train_num.columns]
    max_values_num = [max_values_dict[col] for col in train_num.columns]

    # Inicialización y ajuste del Paso 1 (ExtraTreesRegressor)
    tree_num = ExtraTreesRegressor(n_estimators=50, max_depth=4, min_samples_leaf=3, random_state=1234)
    imputer_num = IterativeImputer(estimator=tree_num, random_state=1234, max_iter=20, 
                                   imputation_order='ascending', initial_strategy='mean',
                                   min_value=min_values_num, max_value=max_values_num)
    
    train_num_imp = pd.DataFrame(imputer_num.fit_transform(train_num), columns=train_num.columns, index=train_num.index)
    test_num_imp  = pd.DataFrame(imputer_num.transform(test_num),     columns=test_num.columns,  index=test_num.index)

    # UNIR Y HACER IMPUTACIÓN CONJUNTA
    train_all = pd.concat([train_num_imp, train_cat], axis=1)
    test_all  = pd.concat([test_num_imp, test_cat], axis=1)

    min_values_all = [min_values_dict[col] for col in train_all.columns]
    max_values_all = [max_values_dict[col] for col in train_all.columns]

    # Inicialización y ajuste del Paso 2 (Imputer conjunto)
    tree_all = ExtraTreesRegressor(n_estimators=50, max_depth=4, min_samples_leaf=3, random_state=1234)
    imputer_all = IterativeImputer(estimator=tree_all, random_state=1234, max_iter=20, 
                                   imputation_order='ascending', initial_strategy='mean',
                                   min_value=min_values_all, max_value=max_values_all)
    
    train_all_imp = imputer_all.fit_transform(train_all)
    test_all_imp  = imputer_all.transform(test_all)

    # CLIPPING Y REDONDEO DE CATEGÓRICAS (ORDINALES)
    # El bucle empieza en el índice donde terminan las numéricas (tal cual tu código original)
    for i in range(train_num.shape[1], train_all.shape[1]):
        train_all_imp[:, i] = np.clip(train_all_imp[:, i], min_values_all[i], max_values_all[i])
        test_all_imp[:, i]  = np.clip(test_all_imp[:, i], min_values_all[i], max_values_all[i])

    train_final = pd.DataFrame(train_all_imp, columns=train_all.columns, index=train_all.index)
    test_final  = pd.DataFrame(test_all_imp, columns=test_all.columns, index=test_all.index)

    for col in train_cat.columns:
        train_final[col] = train_final[col].round().astype(int)
        test_final[col]  = test_final[col].round().astype(int)

    # NOMINALES (MODA Y ONE-HOT)
    if nom:
        m = SimpleImputer(strategy='most_frequent')
        tr_c = pd.DataFrame(m.fit_transform(Xtr[nom]), columns=nom, index=Xtr.index)
        te_c = pd.DataFrame(m.transform(Xte[nom]),     columns=nom, index=Xte.index)
        
        ohe = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')
        tr_oh = ohe.fit_transform(tr_c)
        te_oh = ohe.transform(te_c)
        
        names = ohe.get_feature_names_out(nom)
        tr_c = pd.DataFrame(tr_oh, columns=names, index=Xtr.index)
        te_c = pd.DataFrame(te_oh, columns=names, index=Xte.index)
    else:
        tr_c = pd.DataFrame(index=Xtr.index); te_c = pd.DataFrame(index=Xte.index)

    # COMBINACIÓN FINAL Y ESCALADO
    tr = pd.concat([train_final, tr_c], axis=1)
    te = pd.concat([test_final, te_c], axis=1)
    
    sc = StandardScaler()
    Xtr_s = pd.DataFrame(sc.fit_transform(tr), columns=tr.columns, index=tr.index)
    Xte_s = pd.DataFrame(sc.transform(te),     columns=te.columns, index=te.index)
    
    return Xtr_s, Xte_s, ytr, yte

In [8]:
# Método 1: LASSO stability selection
def sel_lasso_stability(Xtr, ytr, n_iter=80, frac=0.5, C=0.5, umbral=0.6, seed=RANDOM_STATE):
    rng = np.random.RandomState(seed)
    n = len(ytr); m = int(n*frac); cuenta = np.zeros(Xtr.shape[1])
    for _ in range(n_iter):
        idx = rng.choice(n, m, replace=False)
        ys = ytr[idx]
        if len(np.unique(ys)) < 2:
            continue
        lr = LogisticRegression(penalty='l1', solver='liblinear', C=C,
                                class_weight='balanced', max_iter=500)
        lr.fit(Xtr.values[idx], ys)
        cuenta += (np.abs(lr.coef_[0]) > 1e-8)
    freq = cuenta / n_iter
    return [c for c, f in zip(Xtr.columns, freq) if f >= umbral]

# Método 2: Importancia con Random Forest (> media)
def sel_rf_importance(Xtr, ytr, seed=RANDOM_STATE):
    rf = RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=seed, n_jobs=-1)
    rf.fit(Xtr, ytr)
    imp = rf.feature_importances_
    return [c for c, v in zip(Xtr.columns, imp) if v > imp.mean()]

# Método 3: RFECV
def sel_rfecv(Xtr, ytr, seed=RANDOM_STATE):
    cv = StratifiedKFold(5, shuffle=True, random_state=seed)
    r = RFECV(LogisticRegression(class_weight='balanced', max_iter=1000),
              step=1, cv=cv, scoring='f1', min_features_to_select=1, n_jobs=-1)
    r.fit(Xtr, ytr)
    return [c for c, s in zip(Xtr.columns, r.support_) if s]

def seleccion_2_de_3(*conjuntos):
    cnt = Counter()
    for s in conjuntos:
        cnt.update(set(s))
    return sorted([f for f, c in cnt.items() if c >= 2])

In [9]:
# Comparación de 6 clasificadores (GridSearch en train, evaluación en test)
def comparar_clasificadores(Xtr, ytr, Xte, yte):
    pos = int(ytr.sum()); neg = len(ytr) - pos
    spw = neg / max(pos, 1)
    cv = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)
    grids = {
        'SVM': (SVC(class_weight='balanced', random_state=RANDOM_STATE),
                {'C':[0.1,1,10], 'kernel':['linear','rbf'], 'gamma':['scale']}),
        'Random Forest': (RandomForestClassifier(class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1),
                {'n_estimators':[200], 'max_depth':[None,5,10], 'min_samples_split':[2,5]}),
        'Logistic Regression': (LogisticRegression(class_weight='balanced', max_iter=2000, random_state=RANDOM_STATE),
                {'C':[0.1,1,10], 'solver':['lbfgs']}),
        'Decision Tree': (DecisionTreeClassifier(class_weight='balanced', random_state=RANDOM_STATE),
                {'max_depth':[3,5,None], 'min_samples_split':[2,10]}),
        'KNN': (KNeighborsClassifier(weights='distance'),
                {'n_neighbors':[5,7,11], 'metric':['euclidean','manhattan']}),
        'XGBoost': (XGBClassifier(scale_pos_weight=spw, eval_metric='logloss', random_state=RANDOM_STATE),
                {'n_estimators':[100,200], 'max_depth':[3,5], 'learning_rate':[0.05,0.1]}),
    }
    filas = []
    for nombre, (modelo, params) in grids.items():
        gs = GridSearchCV(modelo, params, cv=cv, scoring='f1', n_jobs=-1)
        gs.fit(Xtr, ytr)
        
        # Predicciones finales
        best_model = gs.best_estimator_
        y_pred_tr = best_model.predict(Xtr) # Predicción en TRAIN
        y_pred_te = best_model.predict(Xte) # Predicción en TEST
        
        # Matrices de Confusión
        cm_train = confusion_matrix(ytr, y_pred_tr)
        cm_test  = confusion_matrix(yte, y_pred_te)
        
        # Especificidad (usando la de test para la tabla resumen)
        tn, fp, fn, tp = cm_test.ravel()
        especificidad_te = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        
        try:
            proba = best_model.predict_proba(Xte)[:, 1]
            auc = roc_auc_score(yte, proba)
        except Exception:
            auc = np.nan
            
        filas.append({
            'Modelo': nombre,
            'F1 (test)': round(f1_score(yte, y_pred_te), 4),
            'Precision': round(precision_score(yte, y_pred_te, zero_division=0), 4),
            'Recall': round(recall_score(yte, y_pred_te, zero_division=0), 4),
            'Especificidad': round(especificidad_te, 4),
            'ROC-AUC': round(auc, 4) if not np.isnan(auc) else np.nan,
            'cm_train': cm_train,  # <--- Guardamos la matriz de train
            'cm_test': cm_test,    # <--- Guardamos la matriz de test
            'best_params': gs.best_params_
        })
    return pd.DataFrame(filas).sort_values('F1 (test)', ascending=False).reset_index(drop=True)

## 5. Ejecución sobre los 4 datasets

Para cada dataset: preparamos los datos, aplicamos los 3 métodos de selección (sobre train),
nos quedamos con las variables que aparezcan en ≥ 2 de los 3, y comparamos los clasificadores.

In [10]:
resultados = {}
for nombre, cols in DATASETS.items():
    print('='*70); print('DATASET', nombre); print('='*70)
    Xtr, Xte, ytr, yte = preparar(coh, cols, NOMINALES)
    print(f'  train={Xtr.shape}  test={Xte.shape}  (features tras one-hot: {Xtr.shape[1]})')

    s1 = sel_lasso_stability(Xtr, ytr)
    s2 = sel_rf_importance(Xtr, ytr)
    s3 = sel_rfecv(Xtr, ytr)
    final = seleccion_2_de_3(s1, s2, s3)

    print(f'  LASSO stability ({len(s1)}): {s1}')
    print(f'  RF importance  ({len(s2)}): {s2}')
    print(f'  RFECV          ({len(s3)}): {s3}')
    print(f'  >>> FINAL (>=2/3) ({len(final)}): {final}')

    if len(final) == 0:
        print('  (sin variables seleccionadas; se omite la clasificación)')
        resultados[nombre] = {'final': final, 'tabla': None}
        continue

    tabla = comparar_clasificadores(Xtr[final], ytr, Xte[final], yte)

    # 1. Imprimir la tabla de métricas global del dataset
    print("\nMétricas en Test")
    print(tabla[['Modelo','F1 (test)','Precision','Recall','Especificidad','ROC-AUC']].to_string(index=False))
    
    # 2. Imprimir las matrices de confusión para cada algoritmo de este dataset
    print("\nMatrices de confusión (Train vs Test)")
    for idx, row in tabla.iterrows():
        print(f"\n> MODELO: {row['Modelo']}")
        print(f"  CM Train:\n  {str(row['cm_train']).replace('\n', '\n  ')}")
        print(f"  CM Test:\n  {str(row['cm_test']).replace('\n', '\n  ')}")
        
    print('='*70)
    resultados[nombre] = {'lasso': s1, 'rf': s2, 'rfecv': s3, 'final': final, 'tabla': tabla}

DATASET 1_embarazo


  train=(295, 29)  test=(74, 29)  (features tras one-hot: 29)


  LASSO stability (21): ['aumento_peso_gest', 'talla', 'imc_ini_gest', 'peso_rn', 'apgar_1min', 'apgar_10min', 'tas_1tri', 'tad_1tri', 'edad_materna_gest', 'plgf_MoM', 'etnia_caucasica', 'concepcion_espontanea', 'tomo', 'ratio_sflt1_plgf', 'ant_obstetrico', 'ant_medico', 'trastorno_hipertensivo', 'comp_maternaGrave', 'comp_metabolica', 'tipo_parto_Eutocico', 'tipo_parto_Instrumentado']
  RF importance  (15): ['peso_ini_gest', 'peso_fin_gest', 'aumento_peso_gest', 'talla', 'imc_ini_gest', 'peso_rn', 'eg_parto', 'sflt1_MoM', 'tas_1tri', 'tad_1tri', 'edad_materna_gest', 'plgf_MoM', 'ratio_MoM', 'valor_plgf', 'ratio_sflt1_plgf']
  RFECV          (3): ['peso_ini_gest', 'ratio_MoM', 'ratio_sflt1_plgf']
  >>> FINAL (>=2/3) (11): ['aumento_peso_gest', 'edad_materna_gest', 'imc_ini_gest', 'peso_ini_gest', 'peso_rn', 'plgf_MoM', 'ratio_MoM', 'ratio_sflt1_plgf', 'tad_1tri', 'talla', 'tas_1tri']



Métricas en Test
             Modelo  F1 (test)  Precision  Recall  Especificidad  ROC-AUC
      Random Forest     0.5000     0.4615  0.5455         0.8889   0.8571
            XGBoost     0.5000     0.4615  0.5455         0.8889   0.8701
Logistic Regression     0.4500     0.3103  0.8182         0.6825   0.7792
                SVM     0.4103     0.2857  0.7273         0.6825      NaN
      Decision Tree     0.3784     0.2692  0.6364         0.6984   0.6638
                KNN     0.3158     0.3750  0.2727         0.9206   0.7648

Matrices de confusión (Train vs Test)

> MODELO: Random Forest
  CM Train:
  [[249   0]
   [  0  46]]
  CM Test:
  [[56  7]
   [ 5  6]]

> MODELO: XGBoost
  CM Train:
  [[244   5]
   [  0  46]]
  CM Test:
  [[56  7]
   [ 5  6]]

> MODELO: Logistic Regression
  CM Train:
  [[174  75]
   [ 15  31]]
  CM Test:
  [[43 20]
   [ 2  9]]

> MODELO: SVM
  CM Train:
  [[180  69]
   [ 17  29]]
  CM Test:
  [[43 20]
   [ 3  8]]

> MODELO: Decision Tree
  CM Train:
  [[19

  LASSO stability (8): ['edad_actual', 'imc_actual', 'ant_medico_actual', 'dieta', 'ejercicio', 'estres', 'tabaco_No', 'tabaco_Si']
  RF importance  (5): ['edad_actual', 'peso_actual', 'imc_actual', 'ejercicio', 'estres']
  RFECV          (9): ['edad_actual', 'peso_actual', 'imc_actual', 'ant_medico_actual', 'dieta', 'ejercicio', 'estres', 'tabaco_No', 'tabaco_Si']
  >>> FINAL (>=2/3) (9): ['ant_medico_actual', 'dieta', 'edad_actual', 'ejercicio', 'estres', 'imc_actual', 'peso_actual', 'tabaco_No', 'tabaco_Si']


...

  LASSO stability (15): ['edad_actual', 'ant_medico_actual', 'dieta', 'ejercicio', 'estres', 'grasa_visceral', 'ratio_cintura_cadera', 'hemoglobina_glicada', 'ldl', 'hdl', 'trigliceridos', 'creatinina', 'urato_acidourico', 'tabaco_No', 'tabaco_Si']
  RF importance  (12): ['edad_actual', 'peso_actual', 'imc_actual', 'ejercicio', 'estres', 'grasa_visceral', 'ratio_cintura_cadera', 'ldl', 'hdl', 'trigliceridos', 'creatinina', 'urato_acidourico']
  RFECV          (6): ['edad_actual', 'peso_actual', 'imc_actual', 'ant_medico_actual', 'grasa_visceral', 'tabaco_Si']
  >>> FINAL (>=2/3) (14): ['ant_medico_actual', 'creatinina', 'edad_actual', 'ejercicio', 'estres', 'grasa_visceral', 'hdl', 'imc_actual', 'ldl', 'peso_actual', 'ratio_cintura_cadera', 'tabaco_Si', 'trigliceridos', 'urato_acidourico']



Métricas en Test
             Modelo  F1 (test)  Precision  Recall  Especificidad  ROC-AUC
      Random Forest     0.5385     0.4667  0.6364         0.8730   0.7807
                SVM     0.4737     0.3333  0.8182         0.7143      NaN
Logistic Regression     0.4500     0.3103  0.8182         0.6825   0.8023
            XGBoost     0.4167     0.3846  0.4545         0.8730   0.8038
                KNN     0.3333     0.4286  0.2727         0.9365   0.7244
      Decision Tree     0.2500     0.1905  0.3636         0.7302   0.5079

Matrices de confusión (Train vs Test)

> MODELO: Random Forest
  CM Train:
  [[225  24]
   [  0  46]]
  CM Test:
  [[55  8]
   [ 4  7]]

> MODELO: SVM
  CM Train:
  [[178  71]
   [ 13  33]]
  CM Test:
  [[45 18]
   [ 2  9]]

> MODELO: Logistic Regression
  CM Train:
  [[190  59]
   [ 16  30]]
  CM Test:
  [[43 20]
   [ 2  9]]

> MODELO: XGBoost
  CM Train:
  [[239  10]
   [  1  45]]
  CM Test:
  [[55  8]
   [ 6  5]]

> MODELO: KNN
  CM Train:
  [[249   0]
   

  train=(295, 46)  test=(74, 46)  (features tras one-hot: 46)


  LASSO stability (30): ['aumento_peso_gest', 'talla', 'apgar_1min', 'apgar_10min', 'tas_1tri', 'tad_1tri', 'edad_materna_gest', 'plgf_MoM', 'etnia_caucasica', 'concepcion_espontanea', 'tomo', 'valor_plgf', 'ant_obstetrico', 'ant_medico', 'trastorno_hipertensivo', 'comp_maternaGrave', 'comp_metabolica', 'ant_medico_actual', 'dieta', 'ejercicio', 'estres', 'grasa_visceral', 'ratio_cintura_cadera', 'trigliceridos', 'creatinina', 'urato_acidourico', 'tipo_parto_Eutocico', 'tipo_parto_Instrumentado', 'tabaco_No', 'tabaco_Si']
  RF importance  (29): ['peso_ini_gest', 'peso_fin_gest', 'aumento_peso_gest', 'talla', 'imc_ini_gest', 'peso_rn', 'eg_parto', 'sflt1_MoM', 'tas_1tri', 'tad_1tri', 'edad_materna_gest', 'plgf_MoM', 'ratio_MoM', 'valor_plgf', 'ratio_sflt1_plgf', 'edad_actual', 'peso_actual', 'imc_actual', 'dieta', 'ejercicio', 'estres', 'grasa_visceral', 'ratio_cintura_cadera', 'hemoglobina_glicada', 'ldl', 'hdl', 'trigliceridos', 'creatinina', 'urato_acidourico']
  RFECV          (6): 


Métricas en Test
             Modelo  F1 (test)  Precision  Recall  Especificidad  ROC-AUC
      Random Forest     0.5385     0.4667  0.6364         0.8730   0.8268
            XGBoost     0.5217     0.5000  0.5455         0.9048   0.8543
                SVM     0.4211     0.2963  0.7273         0.6984      NaN
Logistic Regression     0.4000     0.2759  0.7273         0.6667   0.7850
      Decision Tree     0.3333     0.2632  0.4545         0.7778   0.6450
                KNN     0.1667     1.0000  0.0909         1.0000   0.8405

Matrices de confusión (Train vs Test)

> MODELO: Random Forest
  CM Train:
  [[237  12]
   [  1  45]]
  CM Test:
  [[55  8]
   [ 4  7]]

> MODELO: XGBoost
  CM Train:
  [[243   6]
   [  0  46]]
  CM Test:
  [[57  6]
   [ 5  6]]

> MODELO: SVM
  CM Train:
  [[191  58]
   [ 12  34]]
  CM Test:
  [[44 19]
   [ 3  8]]

> MODELO: Logistic Regression
  CM Train:
  [[190  59]
   [ 14  32]]
  CM Test:
  [[42 21]
   [ 3  8]]

> MODELO: Decision Tree
  CM Train:
  [[20

## 6. Resumen comparativo

In [11]:
print('VARIABLES FINALES POR DATASET (aparecen en >=2 de los 3 métodos)\n')
for nombre, r in resultados.items():
    print(f'- {nombre} ({len(r["final"])}): {r["final"]}')

print('\n\nMEJOR CLASIFICADOR POR DATASET\n')
for nombre, r in resultados.items():
    if r.get('tabla') is not None:
        mejor = r['tabla'].iloc[0]
        print(f'- {nombre}: {mejor["Modelo"]} | F1={mejor["F1 (test)"]} | '
              f'Precision={mejor["Precision"]} | Recall={mejor["Recall"]} | ROC-AUC={mejor["ROC-AUC"]}')

VARIABLES FINALES POR DATASET (aparecen en >=2 de los 3 métodos)

- 1_embarazo (11): ['aumento_peso_gest', 'edad_materna_gest', 'imc_ini_gest', 'peso_ini_gest', 'peso_rn', 'plgf_MoM', 'ratio_MoM', 'ratio_sflt1_plgf', 'tad_1tri', 'talla', 'tas_1tri']
- 2_clasicas (9): ['ant_medico_actual', 'dieta', 'edad_actual', 'ejercicio', 'estres', 'imc_actual', 'peso_actual', 'tabaco_No', 'tabaco_Si']
- 3_clasicas_analitica (14): ['ant_medico_actual', 'creatinina', 'edad_actual', 'ejercicio', 'estres', 'grasa_visceral', 'hdl', 'imc_actual', 'ldl', 'peso_actual', 'ratio_cintura_cadera', 'tabaco_Si', 'trigliceridos', 'urato_acidourico']
- 4_completo (17): ['aumento_peso_gest', 'creatinina', 'dieta', 'edad_materna_gest', 'ejercicio', 'estres', 'grasa_visceral', 'plgf_MoM', 'ratio_MoM', 'ratio_cintura_cadera', 'ratio_sflt1_plgf', 'tad_1tri', 'talla', 'tas_1tri', 'trigliceridos', 'urato_acidourico', 'valor_plgf']


MEJOR CLASIFICADOR POR DATASET

- 1_embarazo: Random Forest | F1=0.5 | Precision=0.4615 |

# REUNIÓN 17

- Hacer bucle para ver qué combinación de variables seleccionadas (por lasso, rfimportance, rfecv) maximiza el f1-score. 
- Para las combinaciones de variables que tengan f1-score>0.55 en train, mostrar tabla de métricas (f1-score, especificidad, precision, recall, ROC, matriz de confusión).
- Para las combinaciones de variables que tengan f1-score>0.55, hacer GridSearch para optimizar f1-score y para optimizar especificidad.
- Hacer tabla de comparación (f1-score) (especificidad)

Trabajamos con las variables almacenadas en 'final_x'. La x corresponde al número del dataset.

In [12]:
final_1 = ['aumento_peso_gest', 'edad_materna_gest', 'imc_ini_gest', 'peso_ini_gest', 'peso_rn', 'plgf_MoM', 'ratio_MoM', 'ratio_sflt1_plgf', 'tad_1tri', 'talla', 'tas_1tri']
final_2 = ['ant_medico_actual', 'dieta', 'edad_actual', 'ejercicio', 'estres', 'imc_actual', 'peso_actual', 'tabaco_No', 'tabaco_Si']
final_3 = ['ant_medico_actual', 'creatinina', 'edad_actual', 'estres', 'grasa_visceral', 'hdl', 'imc_actual', 'ldl', 'peso_actual', 'ratio_cintura_cadera', 'tabaco_Si', 'trigliceridos', 'urato_acidourico']
final_4 = ['aumento_peso_gest', 'creatinina', 'edad_materna_gest', 'ejercicio', 'estres', 'grasa_visceral', 'plgf_MoM', 'ratio_MoM', 'ratio_cintura_cadera', 'ratio_sflt1_plgf', 'tad_1tri', 'talla', 'tas_1tri', 'trigliceridos', 'urato_acidourico', 'valor_plgf']

In [13]:
import itertools

def evaluar_6_algoritmos_completo_train(Xtr, Xte, ytr, yte, variables_finales, top_k=3, max_k=None):
    """
    Prueba combinaciones de variables para los 6 algoritmos.
    Calcula F1, Especificidad, Precisión, Recall y ROC-AUC para Train y Test.
    Filtra y selecciona basándose estrictamente en el F1-score de TRAIN completo.

    max_k: tamaño máximo de combinación (nº de variables). None = todas (2^n - 1).
           Topar max_k evita la explosión combinatoria en datasets grandes.
    n_jobs=1 en todos los modelos: con miles de 'fit' sobre datos pequeños, abrir
           pools de hilos (n_jobs=-1) es MÁS lento y puede colgar el kernel.
    """
    pos = int(ytr.sum()); neg = len(ytr) - pos
    spw = neg / max(pos, 1)
    
    # Definición de los 6 modelos con parámetros por defecto (excepto balanceo)
    modelos = {
        'SVM': SVC(class_weight='balanced', probability=True, random_state=RANDOM_STATE),
        'Random Forest': RandomForestClassifier(class_weight='balanced', max_depth=4, n_estimators=100, random_state=RANDOM_STATE, n_jobs=1),
        'Logistic Regression': LogisticRegression(class_weight='balanced', max_iter=2000, solver='liblinear', random_state=RANDOM_STATE),
        'Decision Tree': DecisionTreeClassifier(class_weight='balanced', max_depth=3, random_state=RANDOM_STATE),
        'KNN': KNeighborsClassifier(weights='uniform', n_neighbors=5, n_jobs=1),
        'XGBoost': XGBClassifier(scale_pos_weight=spw, max_depth=3, learning_rate=0.05, eval_metric='logloss', random_state=RANDOM_STATE, n_jobs=1)
    }
    
    # Generar combinaciones hasta tamaño max_k (o todas si max_k es None)
    kmax = len(variables_finales) if max_k is None else min(max_k, len(variables_finales))
    todas_combinaciones = []
    for r in range(1, kmax + 1):
        for combo in itertools.combinations(variables_finales, r):
            todas_combinaciones.append(list(combo))
            
    print(f"  -> Analizando {len(todas_combinaciones)} combinaciones (hasta {kmax} variables) por cada clasificador...")
    resultados_totales = []
    
    # Bucle principal por algoritmo
    for nombre_modelo, modelo in modelos.items():
        filas_modelo = []
        
        for combo in todas_combinaciones:
            Xtr_sub = Xtr[combo]
            Xte_sub = Xte[combo]
            
            # Ajuste del modelo
            modelo.fit(Xtr_sub, ytr)
            
            # Predicciones de clase
            y_pred_tr = modelo.predict(Xtr_sub)
            y_pred_te = modelo.predict(Xte_sub)
            
            # Predicciones de probabilidad para ROC-AUC
            try:
                proba_tr = modelo.predict_proba(Xtr_sub)[:, 1]
                proba_te = modelo.predict_proba(Xte_sub)[:, 1]
                auc_tr = round(roc_auc_score(ytr, proba_tr), 4)
                auc_te = round(roc_auc_score(yte, proba_te), 4)
            except Exception:
                auc_tr = np.nan
                auc_te = np.nan
            
            # Matrices de confusión
            tn_tr, fp_tr, fn_tr, tp_tr = confusion_matrix(ytr, y_pred_tr, labels=[0,1]).ravel()
            tn_te, fp_te, fn_te, tp_te = confusion_matrix(yte, y_pred_te, labels=[0,1]).ravel()
            
            # Métricas en TRAIN
            f1_tr = f1_score(ytr, y_pred_tr, zero_division=0)
            prec_tr = precision_score(ytr, y_pred_tr, zero_division=0)
            rec_tr = recall_score(ytr, y_pred_tr, zero_division=0)
            espec_tr = tn_tr / (tn_tr + fp_tr) if (tn_tr + fp_tr) > 0 else 0.0
            
            # Métricas en TEST
            f1_te = f1_score(yte, y_pred_te, zero_division=0)
            prec_te = precision_score(yte, y_pred_te, zero_division=0)
            rec_te = recall_score(yte, y_pred_te, zero_division=0)
            espec_te = tn_te / (tn_te + fp_te) if (tn_te + fp_te) > 0 else 0.0
            
            filas_modelo.append({
                'Algoritmo': nombre_modelo,
                # Bloque TRAIN
                'F1 (tr)': round(f1_tr, 4),
                'Prec (tr)': round(prec_tr, 4),
                'Rec (tr)': round(rec_tr, 4),
                'Espec (tr)': round(espec_tr, 4),
                'ROC (tr)': auc_tr,
                'CM Train': f'TN={tn_tr} FP={fp_tr} FN={fn_tr} TP={tp_tr}',
                # Bloque TEST
                'F1 (te)': round(f1_te, 4),
                'Prec (te)': round(prec_te, 4),
                'Rec (te)': round(rec_te, 4),
                'Espec (te)': round(espec_te, 4),
                'ROC (te)': auc_te,
                'CM Test': f'TN={tn_te} FP={fp_te} FN={fn_te} TP={tp_te}',
                
                'Num Variables': len(combo),
                'Variables': combo
            })
            
        # Ordenar las combinaciones de este algoritmo específico de mayor a menor F1 de Train
        df_modelo = pd.DataFrame(filas_modelo).sort_values('F1 (tr)', ascending=False).reset_index(drop=True)
        
        # Aplicar el filtro condicional pedido por algoritmo
        df_filtrado = df_modelo[df_modelo['F1 (tr)'] > 0.55].copy()
        
        if len(df_filtrado) == 0:
            # Si no hay ninguna > 0.55, rescatamos el Top 3 absoluto del algoritmo
            df_filtrado = df_modelo.head(top_k).copy()
            df_filtrado['Filtro'] = [f'{i+1}º (Rescate)' for i in range(len(df_filtrado))]
        else:
            # Si hay combinaciones válidas, nos quedamos con su Top 3 óptimo en Train
            df_filtrado = df_filtrado.head(top_k).copy()
            df_filtrado['Filtro'] = [f'{i+1}º (>0.55)' for i in range(len(df_filtrado))]
            
        resultados_totales.append(df_filtrado)
        
    return pd.concat(resultados_totales, ignore_index=True)

In [14]:
import pickle, time

datasets_analisis = {
    '1_embarazo': final_1,
    '2_clasicas': final_2,
    '3_clasicas_analitica': final_3,
    '4_completo': final_4
}

# Búsqueda COMPLETA (sin tope) en D1/D2/D3 — como estaba originalmente.
# Solo el dataset 4 (16 variables = 65.535 combinaciones, ~1.5 h) se topa para que sea viable.
MAX_K = {'1_embarazo': None, '2_clasicas': None, '3_clasicas_analitica': None, '4_completo': 4}

reportes_combos = {}   # reporte (top combos por algoritmo) de cada dataset -> se reutiliza en el GridSearch
splits_datos    = {}   # matrices ya preparadas (Xtr, Xte, ytr, yte) por dataset -> evita recomputar 'preparar'

t_total = time.time()
for nombre_ds, variables_finales in datasets_analisis.items():
    print('='*100)
    print(f"BÚSQUEDA INTEGRAL EN TRAIN: DATASET {nombre_ds.upper()}")
    print('='*100)
    
    # Preparar matrices de datos con tu pipeline
    Xtr, Xte, ytr, yte = preparar(coh, DATASETS[nombre_ds], NOMINALES)
    splits_datos[nombre_ds] = (Xtr, Xte, ytr, yte)
    
    # Ejecutar la evaluación masiva con todas las métricas (con tope de combinación)
    t0 = time.time()
    df_reporte_ds = evaluar_6_algoritmos_completo_train(
        Xtr, Xte, ytr, yte, variables_finales, top_k=3, max_k=MAX_K[nombre_ds])
    reportes_combos[nombre_ds] = df_reporte_ds
    print(f"  (tiempo dataset: {time.time()-t0:.1f}s)")
    
    print(f"\nREPORTES DE MÉTRICAS DETALLADAS POR ALGORITMO ({nombre_ds.upper()})")
    
    # Columnas estructuradas para visualización limpia sin desborde horizontal
    cols_train = ['Filtro', 'F1 (tr)', 'Prec (tr)', 'Rec (tr)', 'Espec (tr)', 'ROC (tr)', 'CM Train']
    cols_test  = ['Filtro', 'F1 (te)', 'Prec (te)', 'Rec (te)', 'Espec (te)', 'ROC (te)', 'CM Test']
    
    for alg, grupo in df_reporte_ds.groupby('Algoritmo'):
        print(f"\n ALGORITMO: {alg}")
        print(f"   Variables usadas en el 1º puesto: {grupo['Variables'].iloc[0]} (Total: {grupo['Num Variables'].iloc[0]})")
        print("\n   [MÉTRICAS CONJUNTO TRAIN]")
        print("   " + grupo[cols_train].to_string(index=False).replace('\n', '\n   '))
        print("\n   [MÉTRICAS CONJUNTO TEST]")
        print("   " + grupo[cols_test].to_string(index=False).replace('\n', '\n   '))
        print("-" * 80)
    print('\n')

print(f"TIEMPO TOTAL BÚSQUEDA: {time.time()-t_total:.1f}s")

# Persistencia: guardamos el reporte para no recomputar el barrido en futuras sesiones
with open('reportes_combos_r16.pkl', 'wb') as f:
    pickle.dump(reportes_combos, f)
print("Guardado -> reportes_combos_r16.pkl")

BÚSQUEDA INTEGRAL EN TRAIN: DATASET 1_EMBARAZO


  -> Analizando 2047 combinaciones (hasta 11 variables) por cada clasificador...


  (tiempo dataset: 183.4s)

REPORTES DE MÉTRICAS DETALLADAS POR ALGORITMO (1_EMBARAZO)

 ALGORITMO: Decision Tree
   Variables usadas en el 1º puesto: ['imc_ini_gest', 'peso_ini_gest', 'plgf_MoM', 'ratio_MoM', 'ratio_sflt1_plgf', 'tas_1tri'] (Total: 6)

   [MÉTRICAS CONJUNTO TRAIN]
       Filtro  F1 (tr)  Prec (tr)  Rec (tr)  Espec (tr)  ROC (tr)                 CM Train
   1º (>0.55)   0.5556     0.4375    0.7609      0.8193    0.8249 TN=204 FP=45 FN=11 TP=35
   2º (>0.55)   0.5556     0.4375    0.7609      0.8193    0.8249 TN=204 FP=45 FN=11 TP=35
   3º (>0.55)   0.5556     0.4375    0.7609      0.8193    0.8249 TN=204 FP=45 FN=11 TP=35

   [MÉTRICAS CONJUNTO TEST]
       Filtro  F1 (te)  Prec (te)  Rec (te)  Espec (te)  ROC (te)               CM Test
   1º (>0.55)   0.3571     0.2941    0.4545      0.8095    0.6443 TN=51 FP=12 FN=6 TP=5
   2º (>0.55)   0.3571     0.2941    0.4545      0.8095    0.6443 TN=51 FP=12 FN=6 TP=5
   3º (>0.55)   0.3571     0.2941    0.4545      0.8095    0

  (tiempo dataset: 44.8s)

REPORTES DE MÉTRICAS DETALLADAS POR ALGORITMO (2_CLASICAS)

 ALGORITMO: Decision Tree
   Variables usadas en el 1º puesto: ['ant_medico_actual', 'dieta', 'ejercicio', 'estres', 'imc_actual', 'peso_actual', 'tabaco_Si'] (Total: 7)

   [MÉTRICAS CONJUNTO TRAIN]
         Filtro  F1 (tr)  Prec (tr)  Rec (tr)  Espec (tr)  ROC (tr)                 CM Train
   1º (Rescate)   0.5227     0.5476       0.5      0.9237    0.7925 TN=230 FP=19 FN=23 TP=23
   2º (Rescate)   0.5227     0.5476       0.5      0.9237    0.7925 TN=230 FP=19 FN=23 TP=23
   3º (Rescate)   0.5227     0.5476       0.5      0.9237    0.7925 TN=230 FP=19 FN=23 TP=23

   [MÉTRICAS CONJUNTO TEST]
         Filtro  F1 (te)  Prec (te)  Rec (te)  Espec (te)  ROC (te)               CM Test
   1º (Rescate)   0.3077     0.2667    0.3636      0.8254    0.5404 TN=52 FP=11 FN=7 TP=4
   2º (Rescate)   0.3077     0.2667    0.3636      0.8254    0.5404 TN=52 FP=11 FN=7 TP=4
   3º (Rescate)   0.3077     0.2667    0.3

  -> Analizando 8191 combinaciones (hasta 13 variables) por cada clasificador...


  (tiempo dataset: 741.5s)

REPORTES DE MÉTRICAS DETALLADAS POR ALGORITMO (3_CLASICAS_ANALITICA)

 ALGORITMO: Decision Tree
   Variables usadas en el 1º puesto: ['ant_medico_actual', 'estres', 'hdl', 'ldl', 'tabaco_Si', 'trigliceridos', 'urato_acidourico'] (Total: 7)

   [MÉTRICAS CONJUNTO TRAIN]
         Filtro  F1 (tr)  Prec (tr)  Rec (tr)  Espec (tr)  ROC (tr)                 CM Train
   1º (Rescate)   0.5421     0.4754    0.6304      0.8715    0.7895 TN=217 FP=32 FN=17 TP=29
   2º (Rescate)   0.5421     0.4754    0.6304      0.8715    0.7895 TN=217 FP=32 FN=17 TP=29
   3º (Rescate)   0.5421     0.4754    0.6304      0.8715    0.7895 TN=217 FP=32 FN=17 TP=29

   [MÉTRICAS CONJUNTO TEST]
         Filtro  F1 (te)  Prec (te)  Rec (te)  Espec (te)  ROC (te)               CM Test
   1º (Rescate)   0.1379     0.1111    0.1818      0.7460    0.4921 TN=47 FP=16 FN=9 TP=2
   2º (Rescate)   0.1290     0.1000    0.1818      0.7143    0.4646 TN=45 FP=18 FN=9 TP=2
   3º (Rescate)   0.1290     0.

  -> Analizando 2516 combinaciones (hasta 4 variables) por cada clasificador...


  (tiempo dataset: 220.2s)

REPORTES DE MÉTRICAS DETALLADAS POR ALGORITMO (4_COMPLETO)

 ALGORITMO: Decision Tree
   Variables usadas en el 1º puesto: ['creatinina', 'grasa_visceral', 'plgf_MoM', 'tas_1tri'] (Total: 4)

   [MÉTRICAS CONJUNTO TRAIN]
       Filtro  F1 (tr)  Prec (tr)  Rec (tr)  Espec (tr)  ROC (tr)                 CM Train
   1º (>0.55)   0.5714     0.5085    0.6522      0.8835    0.7876 TN=220 FP=29 FN=16 TP=30
   2º (>0.55)   0.5688     0.4921    0.6739      0.8715    0.7859 TN=217 FP=32 FN=15 TP=31
   3º (>0.55)   0.5686     0.5179    0.6304      0.8916    0.8243 TN=222 FP=27 FN=17 TP=29

   [MÉTRICAS CONJUNTO TEST]
       Filtro  F1 (te)  Prec (te)  Rec (te)  Espec (te)  ROC (te)               CM Test
   1º (>0.55)   0.4286     0.3529    0.5455      0.8254    0.6804 TN=52 FP=11 FN=5 TP=6
   2º (>0.55)   0.4167     0.3846    0.4545      0.8730    0.6551  TN=55 FP=8 FN=6 TP=5
   3º (>0.55)   0.4000     0.4444    0.3636      0.9206    0.6349  TN=58 FP=5 FN=7 TP=4
------

## Optimización de hiperparámetros (GridSearch) sobre la mejor combinación

**Idea (optimización en dos etapas):**
1. *Etapa 1 — qué variables* (ya hecha arriba): la búsqueda integral encontró, **por algoritmo**, la
   combinación de variables con mejor F1 en train. Esos modelos usaban **hiperparámetros por defecto**.
2. *Etapa 2 — qué hiperparámetros* (esta celda): **fijamos** esa mejor combinación de variables y
   lanzamos `GridSearchCV(scoring='f1')` para afinar los hiperparámetros del modelo.

Se hace solo para **SVM**, **Regresión Logística** y el **árbol (Decision/Random)** — como pide el
enunciado. Para "Decision/Random" se optimizan **los dos** (Decision Tree y Random Forest) y la tabla
marca cuál gana. Comparamos: **F1 en test SIN grid** (Etapa 1) · **F1 por CV en train** (lo que el grid
optimiza, sin tocar el test) · **F1 en test CON grid**.


In [15]:
# Rejilla de hiperparámetros por algoritmo (todos con balanceo de clases)
def grid_de(nombre):
    if nombre == 'SVM':
        est  = SVC(class_weight='balanced', probability=True, random_state=RANDOM_STATE)
        grid = {'C': [0.01, 0.1, 1, 10, 100], 'kernel': ['linear', 'rbf'], 'gamma': ['scale', 'auto']}
    elif nombre == 'Logistic Regression':
        est  = LogisticRegression(class_weight='balanced', max_iter=2000, solver='liblinear', random_state=RANDOM_STATE)
        grid = {'C': [0.01, 0.1, 1, 10, 100], 'penalty': ['l1', 'l2']}
    elif nombre == 'Decision Tree':
        est  = DecisionTreeClassifier(class_weight='balanced', random_state=RANDOM_STATE)
        grid = {'max_depth': [2, 3, 4, 5, None], 'min_samples_split': [2, 5, 10],
                'min_samples_leaf': [1, 3, 5], 'criterion': ['gini', 'entropy']}
    elif nombre == 'Random Forest':
        est  = RandomForestClassifier(class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)
        grid = {'n_estimators': [100, 200, 300], 'max_depth': [3, 5, 8, None],
                'min_samples_split': [2, 5], 'min_samples_leaf': [1, 3]}
    return est, grid

ALGORITMOS_GRID = ['SVM', 'Logistic Regression', 'Decision Tree', 'Random Forest']
cv = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)

resumen_grid = {}
for nombre_ds in datasets_analisis:
    print('=' * 100)
    print(f"GRIDSEARCH SOBRE LA MEJOR COMBINACIÓN — DATASET {nombre_ds.upper()}")
    print('=' * 100)
    Xtr, Xte, ytr, yte = splits_datos[nombre_ds]
    df_rep = reportes_combos[nombre_ds]

    filas = []
    for alg in ALGORITMOS_GRID:
        sub = df_rep[df_rep['Algoritmo'] == alg]
        if len(sub) == 0:
            continue
        mejor = sub.iloc[0]               # 1er puesto = mejor combinación por F1 de train (Etapa 1)
        combo = mejor['Variables']
        f1_te_sin_grid = mejor['F1 (te)']  # F1 en test del modelo por defecto (Etapa 1)

        est, grid = grid_de(alg)
        gs = GridSearchCV(est, grid, cv=cv, scoring='f1', n_jobs=-1)
        gs.fit(Xtr[combo], ytr)
        best = gs.best_estimator_

        y_pred_te = best.predict(Xte[combo])
        tn, fp, fn, tp = confusion_matrix(yte, y_pred_te, labels=[0, 1]).ravel()
        espec_te = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        try:
            auc_te = round(roc_auc_score(yte, best.predict_proba(Xte[combo])[:, 1]), 4)
        except Exception:
            auc_te = np.nan

        filas.append({
            'Algoritmo': alg,
            'Nº vars': len(combo),
            'F1 test SIN grid': round(f1_te_sin_grid, 4),
            'F1 CV-train (grid)': round(gs.best_score_, 4),
            'F1 test CON grid': round(f1_score(yte, y_pred_te, zero_division=0), 4),
            'Espec test': round(espec_te, 4),
            'Recall test': round(recall_score(yte, y_pred_te, zero_division=0), 4),
            'ROC test': auc_te,
            'CM test': f'TN={tn} FP={fp} FN={fn} TP={tp}',
            'best_params': gs.best_params_,
            'Variables': combo,
        })

    tabla_grid = pd.DataFrame(filas)
    resumen_grid[nombre_ds] = tabla_grid

    cols_show = ['Algoritmo', 'Nº vars', 'F1 test SIN grid', 'F1 CV-train (grid)',
                 'F1 test CON grid', 'Espec test', 'Recall test', 'ROC test', 'CM test']
    print(tabla_grid[cols_show].to_string(index=False))

    # "Decision/Random": entre los dos árboles, cuál gana (por F1 en test con grid)
    arboles = tabla_grid[tabla_grid['Algoritmo'].isin(['Decision Tree', 'Random Forest'])]
    if len(arboles):
        gana = arboles.sort_values('F1 test CON grid', ascending=False).iloc[0]
        print(f"\n  >> Mejor árbol (Decision/Random): {gana['Algoritmo']} "
              f"| F1 test con grid = {gana['F1 test CON grid']}")

    print("\n  Hiperparámetros óptimos (best_params) por algoritmo:")
    for _, r in tabla_grid.iterrows():
        print(f"   - {r['Algoritmo']:<20}: {r['best_params']}")
        print(f"     variables ({r['Nº vars']}): {r['Variables']}")
    print('\n')

GRIDSEARCH SOBRE LA MEJOR COMBINACIÓN — DATASET 1_EMBARAZO


          Algoritmo  Nº vars  F1 test SIN grid  F1 CV-train (grid)  F1 test CON grid  Espec test  Recall test  ROC test               CM test
                SVM        8            0.5161              0.3945            0.4737      0.7143       0.8182    0.7879 TN=45 FP=18 FN=2 TP=9
Logistic Regression        7            0.3429              0.4242            0.3429      0.7143       0.5455    0.6652 TN=45 FP=18 FN=5 TP=6
      Decision Tree        6            0.3571              0.3864            0.3571      0.8095       0.4545    0.6443 TN=51 FP=12 FN=6 TP=5
      Random Forest        9            0.5714              0.3444            0.5833      0.9048       0.6364    0.7893  TN=57 FP=6 FN=4 TP=7

  >> Mejor árbol (Decision/Random): Random Forest | F1 test con grid = 0.5833

  Hiperparámetros óptimos (best_params) por algoritmo:
   - SVM                 : {'C': 0.01, 'gamma': 'scale', 'kernel': 'linear'}
     variables (8): ['aumento_peso_gest', 'edad_materna_gest', 'imc_ini_gest',

          Algoritmo  Nº vars  F1 test SIN grid  F1 CV-train (grid)  F1 test CON grid  Espec test  Recall test  ROC test               CM test
                SVM        9            0.3077              0.4009            0.4091      0.6190       0.8182    0.7864 TN=39 FP=24 FN=2 TP=9
Logistic Regression        7            0.4286              0.4249            0.4286      0.6508       0.8182    0.7821 TN=41 FP=22 FN=2 TP=9
      Decision Tree        7            0.3077              0.3049            0.3158      0.4127       0.8182    0.6912 TN=26 FP=37 FN=2 TP=9
      Random Forest        7            0.3333              0.3885            0.2963      0.8095       0.3636    0.7229 TN=51 FP=12 FN=7 TP=4

  >> Mejor árbol (Decision/Random): Decision Tree | F1 test con grid = 0.3158

  Hiperparámetros óptimos (best_params) por algoritmo:
   - SVM                 : {'C': 10, 'gamma': 'scale', 'kernel': 'linear'}
     variables (9): ['ant_medico_actual', 'dieta', 'edad_actual', 'ejercicio', '

          Algoritmo  Nº vars  F1 test SIN grid  F1 CV-train (grid)  F1 test CON grid  Espec test  Recall test  ROC test                CM test
                SVM       11            0.3571              0.3344            0.3043      0.5556       0.6364    0.7042  TN=35 FP=28 FN=4 TP=7
Logistic Regression        8            0.3902              0.3855            0.3810      0.6349       0.7273    0.7489  TN=40 FP=23 FN=3 TP=8
      Decision Tree        7            0.1379              0.2789            0.0571      0.6349       0.0909    0.3600 TN=40 FP=23 FN=10 TP=1
      Random Forest        8            0.3226              0.3195            0.4000      0.8571       0.4545    0.7460   TN=54 FP=9 FN=6 TP=5

  >> Mejor árbol (Decision/Random): Random Forest | F1 test con grid = 0.4

  Hiperparámetros óptimos (best_params) por algoritmo:
   - SVM                 : {'C': 0.1, 'gamma': 'auto', 'kernel': 'rbf'}
     variables (11): ['ant_medico_actual', 'creatinina', 'edad_actual', 'estres',

          Algoritmo  Nº vars  F1 test SIN grid  F1 CV-train (grid)  F1 test CON grid  Espec test  Recall test  ROC test               CM test
                SVM        4            0.5333              0.4032            0.3571      0.8095       0.4545    0.7460 TN=51 FP=12 FN=6 TP=5
Logistic Regression        4            0.3684              0.4179            0.3684      0.6825       0.6364    0.7056 TN=43 FP=20 FN=4 TP=7
      Decision Tree        4            0.4286              0.4197            0.3556      0.5873       0.7273    0.7186 TN=37 FP=26 FN=3 TP=8
      Random Forest        4            0.6400              0.3645            0.6316      0.9683       0.5455    0.8528  TN=61 FP=2 FN=5 TP=6

  >> Mejor árbol (Decision/Random): Random Forest | F1 test con grid = 0.6316

  Hiperparámetros óptimos (best_params) por algoritmo:
   - SVM                 : {'C': 10, 'gamma': 'scale', 'kernel': 'rbf'}
     variables (4): ['aumento_peso_gest', 'grasa_visceral', 'trigliceridos', 'valor